In [10]:
using ITensors
using ITensorMPS
using Random
using LinearAlgebra
using Statistics
using PrettyTables

#=
    Generates the MPO for the EHM Hamiltonian 
    with strengths J, U and V. 
    Requires a SiteType sites.
=#
function H_EHM(N, J, U, V, sites)
    os = OpSum()
    for i in 1:(N - 1)
      # Knetic 
      os -= J, "Cdagup", i, "Cup", i + 1
      os -= J, "Cdagup", i + 1, "Cup", i
      os -= J, "Cdagdn", i, "Cdn", i + 1
      os -= J, "Cdagdn", i + 1, "Cdn", i
      # Nearest-neighbours
      os += V, "Ntot", i, "Ntot", i + 1
    end
    # on-site
    for i in 1:N
      os += U, "Nupdn", i
    end
    return MPO(os, sites)
end

function random_metallic_state(L, Nup, Ndn)
    state = fill("Emp", L)
    p = Nup + Ndn
    for i in 1:L
        j = L - i
        if(p > j)
            state[j] = "UpDn"
            p -= 2
        elseif (p > 0) 
            state[j] = j % 2 == 1 ? "Up" : "Dn"
            p -= 1
        end
    end
    return state
end

function random_cdw_state(L, Nup, Ndn)
    state = fill("Emp", L)
    Nup_extra = Int.(Nup % Ndn)
    for i in 1:2:(L - Nup_extra)
        state[i] = "UpDn"
        Nup-=1
    end
    if Nup != 0
        state[L] = "Up"
    end
    return state
end

random_cdw_state (generic function with 1 method)

In [19]:
function density_operators(N, psi, sites)
    upd = fill(0.0, N)
    dnd = fill(0.0, N)
    updn = fill(0.0, N) 
    for j in 1:N
        orthogonalize!(psi, j)
        psidag_j = dag(prime(psi[j], "Site"))
        upd[j] = scalar(psidag_j * op(sites, "Nup", j) * psi[j])
        dnd[j] = scalar(psidag_j * op(sites, "Ndn", j) * psi[j])
        updn[j] = scalar(psidag_j * op(sites, "Nupdn", j) * psi[j])
    end
    return upd, dnd, updn
end

function average_single_site_entanglement(L, up, dn, updn)
    single_site_entanglement = fill(0.0, L)
    for i in 1:L 
        w_2 = updn[i] 
        w_up = up[i] - w_2 
        w_dn = dn[i] - w_2 
        w_0 = 1 - w_up - w_dn - w_2 
        single_site_entanglement[i] = 1 - (w_2^2 + w_up^2 + w_dn^2 + w_0^2)
    end
    return Statistics.mean(single_site_entanglement)
end 
function S(rdm, L)

    println("trace (rho) = ", tr(rdm))
    println("ishermitian(rho) = ", ishermitian(rdm))

    lambdas = eigvals(rdm)
    S_bit = 0
    S = 0
    #=
    println("eigenvals = ", lambdas)
    println("eigenvals / L= ", lambdas / L)
    println("sum(eigvals) = ", sum(real(lambdas)))
    =#
    for lambda in lambdas
        lambda = real(lambda) 
        # S -= lambda * log2(lambda)
        # CRITICAL: Handle lambda <= 0 for log2
        if lambda > 1e-15 # A small threshold to avoid errors with log2
            S_bit -= lambda * log2(lambda)
            S -= lambda * log(lambda)
        end
    end 
    return S, S_bit
end

function build_1_particle_rdm(state) 
    L = length(state)
    #=  
        N and not L since any site can have spin up or down
    =#
    rho_1 = zeros(ComplexF64, 2*L, 2*L) 

    Cupup = correlation_matrix(state, "Cdagup", "Cup")
    Cdndn = correlation_matrix(state, "Cdagdn", "Cdn")
    Cupdn = correlation_matrix(state, "Cdagup", "Cdn")
    Cdnup = correlation_matrix(state, "Cdagdn", "Cup")   
    
    @pt Cupdn
    @pt Cdnup

    for i in 1:L
        for j in 1:L
            # Blocks of the correlation matrix.
            i_up = 2 * (i-1) + 1 
            i_dn = 2 * (i-1) + 2
            j_up = 2 * (j-1) + 1
            j_dn = 2 * (j-1) + 2

            rho_1[i_up, j_up] = Cupup[i,j]
            rho_1[i_up, j_dn] = Cupdn[i,j]
            rho_1[i_dn, j_up] = Cdnup[i,j]
            rho_1[i_dn, j_dn] = Cdndn[i,j]
        end 
    end

    @show tr(rho_1) 
    rho_1 = rho_1 / L 
    @show tr(rho_1)
    return rho_1
end

build_1_particle_rdm (generic function with 1 method)

In [64]:
L = 4

Npart = floor(Int, L/2) 
Nup = Npart + L % 2 
Ndn = L - Nup 

sites = siteinds("Electron", L; conserve_qns=true)

J = 1.0 
U = -8.0
V = 6.0

H = H_EHM(L, J, U, V, sites)

state = random_cdw_state(L, Nup, Ndn)

psi0 = random_mps(sites, state; linkdims=4)

nsweeps = 20
maxdim = [50, 100, 200, 400, 800, 800, 1000, 1200]
cutoff = [1E-14]

energy, psi = dmrg(H, psi0; nsweeps, maxdim, cutoff)

After sweep 1 energy=-16.410540765489593  maxlinkdim=16 maxerr=0.00E+00 time=0.015
After sweep 2 energy=-16.44207118791628  maxlinkdim=16 maxerr=0.00E+00 time=0.010
After sweep 3 energy=-16.462239239694213  maxlinkdim=16 maxerr=0.00E+00 time=0.021
After sweep 4 energy=-16.47359099874226  maxlinkdim=16 maxerr=0.00E+00 time=0.011
After sweep 5 energy=-16.479538476843565  maxlinkdim=16 maxerr=0.00E+00 time=0.019
After sweep 6 energy=-16.482558811373544  maxlinkdim=16 maxerr=0.00E+00 time=0.014
After sweep 7 energy=-16.484074577834168  maxlinkdim=16 maxerr=0.00E+00 time=0.016
After sweep 8 energy=-16.48483119453265  maxlinkdim=16 maxerr=0.00E+00 time=0.010
After sweep 9 energy=-16.48520771624911  maxlinkdim=16 maxerr=0.00E+00 time=0.016
After sweep 10 energy=-16.485394750665467  maxlinkdim=16 maxerr=0.00E+00 time=0.016
After sweep 11 energy=-16.485487564226734  maxlinkdim=16 maxerr=0.00E+00 time=0.011
After sweep 12 energy=-16.485533596483577  maxlinkdim=16 maxerr=0.00E+00 time=0.015
After

(-16.485578691913723, MPS
[1] ((dim=4|id=795|"Electron,Site,n=1") <Out>
 1: QN(("Nf",0,-1),("Sz",0)) => 1
 2: QN(("Nf",1,-1),("Sz",1)) => 1
 3: QN(("Nf",1,-1),("Sz",-1)) => 1
 4: QN(("Nf",2,-1),("Sz",0)) => 1, (dim=4|id=725|"Link,l=1") <Out>
 1: QN(("Nf",2,-1),("Sz",0)) => 1
 2: QN(("Nf",3,-1),("Sz",-1)) => 1
 3: QN(("Nf",3,-1),("Sz",1)) => 1
 4: QN(("Nf",4,-1),("Sz",0)) => 1)
[2] ((dim=16|id=488|"Link,l=2") <Out>
 1: QN(("Nf",0,-1),("Sz",0)) => 1
 2: QN(("Nf",1,-1),("Sz",-1)) => 2
 3: QN(("Nf",1,-1),("Sz",1)) => 2
 4: QN(("Nf",2,-1),("Sz",-2)) => 1
 5: QN(("Nf",2,-1),("Sz",0)) => 4
 6: QN(("Nf",2,-1),("Sz",2)) => 1
 7: QN(("Nf",3,-1),("Sz",-1)) => 2
 8: QN(("Nf",3,-1),("Sz",1)) => 2
 9: QN(("Nf",4,-1),("Sz",0)) => 1, (dim=4|id=771|"Electron,Site,n=2") <Out>
 1: QN(("Nf",0,-1),("Sz",0)) => 1
 2: QN(("Nf",1,-1),("Sz",1)) => 1
 3: QN(("Nf",1,-1),("Sz",-1)) => 1
 4: QN(("Nf",2,-1),("Sz",0)) => 1, (dim=4|id=725|"Link,l=1") <In>
 1: QN(("Nf",2,-1),("Sz",0)) => 1
 2: QN(("Nf",3,-1),("Sz",-1)

In [65]:
rho_1 = build_1_particle_rdm(psi)

┌────────┬────────┬────────┬────────┐
│ Col. 1 │ Col. 2 │ Col. 3 │ Col. 4 │
├────────┼────────┼────────┼────────┤
│    0.0 │    0.0 │    0.0 │    0.0 │
│    0.0 │    0.0 │    0.0 │    0.0 │
│    0.0 │    0.0 │    0.0 │    0.0 │
│    0.0 │    0.0 │    0.0 │    0.0 │
└────────┴────────┴────────┴────────┘
┌────────┬────────┬────────┬────────┐
│ Col. 1 │ Col. 2 │ Col. 3 │ Col. 4 │
├────────┼────────┼────────┼────────┤
│    0.0 │    0.0 │    0.0 │    0.0 │
│    0.0 │    0.0 │    0.0 │    0.0 │
│    0.0 │    0.0 │    0.0 │    0.0 │
│    0.0 │    0.0 │    0.0 │    0.0 │
└────────┴────────┴────────┴────────┘
tr(rho_1) = 3.999999999999999 + 0.0im
tr(rho_1) = 0.9999999999999998 + 0.0im


8×8 Matrix{ComplexF64}:
     0.185136+0.0im           0.0+0.0im  …           0.0+0.0im
          0.0+0.0im      0.185135+0.0im     -0.000428281+0.0im
    0.0273372+0.0im           0.0+0.0im              0.0+0.0im
          0.0+0.0im     0.0273372+0.0im       0.00116778+0.0im
   0.00116483+0.0im           0.0+0.0im              0.0+0.0im
          0.0+0.0im    0.00116404+0.0im  …     0.0273784+0.0im
 -0.000428407+0.0im           0.0+0.0im              0.0+0.0im
          0.0+0.0im  -0.000428281+0.0im         0.184806+0.0im

In [58]:
@pt real(rho_1)

┌──────────────┬────────────┬──────────────┬─────────────┬──────────────┬────────────┬──────────────┬─────────────┬──────────────┬────────────┐
│       Col. 1 │     Col. 2 │       Col. 3 │      Col. 4 │       Col. 5 │     Col. 6 │       Col. 7 │      Col. 8 │       Col. 9 │    Col. 10 │
├──────────────┼────────────┼──────────────┼─────────────┼──────────────┼────────────┼──────────────┼─────────────┼──────────────┼────────────┤
│      0.19934 │        0.0 │     0.010922 │         0.0 │ -0.000739089 │        0.0 │ -0.000100896 │         0.0 │   8.08177e-6 │        0.0 │
│          0.0 │   0.159455 │          0.0 │   0.0111901 │          0.0 │  0.0684644 │          0.0 │  0.00108267 │          0.0 │ -0.0402063 │
│     0.010922 │        0.0 │   0.00165595 │         0.0 │    0.0134095 │        0.0 │  0.000973148 │         0.0 │ -0.000101099 │        0.0 │
│          0.0 │  0.0111901 │          0.0 │ 0.000922296 │          0.0 │ 0.00709239 │          0.0 │ 0.000372629 │          0.0 │ 0.001

In [66]:
E_p, _ = S(rho_1, L)

E_p = E_p - log(L)

@show E_p

trace (rho) = 0.9999999999999998 + 0.0im
ishermitian(rho) = true
E_p = 0.546010901687324


0.546010901687324

In [69]:
"""
    build_bulk_rho_1(Cupup, Cdndn, Cupdn, Cdnup, up, dn; L, bulk_range=nothing)

Construct the bulk-restricted one-particle reduced density matrix (rho_1^bulk)
from the correlation matrices and local densities.

Arguments:
- Cupup, Cdndn, Cupdn, Cdnup: L×L correlation matrices
- up, dn: arrays of length L with ⟨n_up⟩ and ⟨n_dn⟩
- L: total number of sites

Keyword arguments:
- bulk_range: an integer range like `17:48`; defaults to `L÷4+1 : 3L÷4`

Returns:
- rho_bulk: (2L_b × 2L_b) normalized particle-reduced density matrix
- N_bulk: total number of particles in the bulk
"""
function build_bulk_rho_1(Cupup, Cdndn, Cupdn, Cdnup, up, dn; L, bulk_range=nothing)
    
    if isnothing(bulk_range)
        bulk_range = (L ÷ 4 + 1):(3L ÷ 4)
    end

    @show bulk_range

    bulk_sites = collect(bulk_range)
    L_b = length(bulk_sites)

    rho_bulk = zeros(ComplexF64, 2L_b, 2L_b)

    for (bi, i) in enumerate(bulk_sites)
        for (bj, j) in enumerate(bulk_sites)
            i_up = 2 * (bi - 1) + 1
            i_dn = 2 * (bi - 1) + 2
            j_up = 2 * (bj - 1) + 1
            j_dn = 2 * (bj - 1) + 2

            rho_bulk[i_up, j_up] = Cupup[i, j]
            rho_bulk[i_dn, j_dn] = Cdndn[i, j]
            rho_bulk[i_up, j_dn] = Cupdn[i, j]
            rho_bulk[i_dn, j_up] = Cdnup[i, j]
        end
    end

    # Total number of particles in the bulk
    N_bulk = sum(up[bulk_sites]) + sum(dn[bulk_sites])

    # Normalize so Tr[rho_bulk] = 1
    rho_bulk ./= N_bulk

    return rho_bulk, N_bulk
end


build_bulk_rho_1

In [71]:
Cupup = correlation_matrix(psi, "Cdagup", "Cup")
Cdndn = correlation_matrix(psi, "Cdagdn", "Cdn")
Cupdn = correlation_matrix(psi, "Cdagup", "Cdn")
Cdnup = correlation_matrix(psi, "Cdagdn", "Cup")   

up, dn, _ = density_operators(L, psi, sites)

rho_bulk, N_bulk = build_bulk_rho_1(Cupup, Cdndn, Cupdn, Cdnup, up, dn; L=L)
vals = eigvals(rho_bulk)
S_bulk = -sum(λ -> λ > 0 ? λ * log2(λ) : 0.0, vals)
E_p_bulk = S_bulk - log2(N_bulk)

bulk_range = 2:3


1.9384048306603405